<a href="https://colab.research.google.com/github/salia40/mini-project-salia/blob/main/mini%20project%20web%205.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import subprocess
import time
import re

print("--- [1/5] Menginstal PHP, Composer, MySQL, & Cloudflare Tunnel ---")
os.system("apt-get update -qq")
os.system("apt-get install mysql-server php php-cli php-mysql php-xml php-mbstring php-curl unzip wget -y -qq")

# Install Composer
os.system("curl -sS https://getcomposer.org/installer | php -- --install-dir=/usr/local/bin --filename=composer")

# Download Cloudflare Tunnel
os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
os.system("chmod +x cloudflared")

# Start MySQL
os.system("service mysql start")
os.system("mysql -u root -e 'CREATE DATABASE IF NOT EXISTS inventory_laravel;'")

print("--- [2/5] Membuat Project Laravel (inventory-laravel) ---")
if not os.path.exists("inventory-laravel"):
    os.system("composer create-project laravel/laravel inventory-laravel --prefer-dist -qq")

os.chdir("inventory-laravel")

# Setup .env
env_content = """
APP_NAME=AksesorisInventory
APP_ENV=local
APP_KEY=base64:vI6sY/x2+o4/O4E5m0Z4w/Y6Z9sQ2jN4S4rX2l9W5sM=
APP_DEBUG=true
APP_URL=http://localhost:8000

DB_CONNECTION=mysql
DB_HOST=127.0.0.1
DB_PORT=3306
DB_DATABASE=inventory_laravel
DB_USERNAME=root
DB_PASSWORD=
"""
with open(".env", "w") as f:
    f.write(env_content.strip())

os.system("php artisan key:generate")

print("--- [3/5] Membuat Migration & Model ---")

# Migration Categories & Products
mig_categories = """<?php
use Illuminate\\Database\\Migrations\\Migration;
use Illuminate\\Database\\Schema\\Blueprint;
use Illuminate\\Support\\Facades\\Schema;

return new class extends Migration {
    public function up(): void {
        Schema::create('categories', function (Blueprint $table) {
            $table->id();
            $table->string('name');
            $table->timestamps();
        });

        Schema::create('products', function (Blueprint $table) {
            $table->id();
            $table->foreignId('category_id')->constrained()->restrictOnDelete();
            $table->string('sku')->unique();
            $table->string('name');
            $table->decimal('price', 12, 2);
            $table->integer('stock')->default(0);
            $table->integer('minimum_stock')->default(5);
            $table->timestamps();
        });
    }

    public function down(): void {
        Schema::dropIfExists('products');
        Schema::dropIfExists('categories');
    }
};
"""
with open("database/migrations/2026_10_13_000001_create_categories_and_products_tables.php", "w") as f:
    f.write(mig_categories)

# Model Category
model_cat = """<?php
namespace App\\Models;

use Illuminate\\Database\\Eloquent\\Factories\\HasFactory;
use Illuminate\\Database\\Eloquent\\Model;

class Category extends Model {
    use HasFactory;
    protected $fillable = ['name'];

    public function products() {
        return $this->hasMany(Product::class);
    }
}
"""
with open("app/Models/Category.php", "w") as f:
    f.write(model_cat)

# Model Product
model_prod = """<?php
namespace App\\Models;

use Illuminate\\Database\\Eloquent\\Factories\\HasFactory;
use Illuminate\\Database\\Eloquent\\Model;

class Product extends Model {
    use HasFactory;
    protected $fillable = ['category_id', 'sku', 'name', 'price', 'stock', 'minimum_stock'];

    public function category() {
        return $this->belongsTo(Category::class);
    }
}
"""
with open("app/Models/Product.php", "w") as f:
    f.write(model_prod)

# Database Seeder
seeder_code = """<?php
namespace Database\\Seeders;

use Illuminate\\Database\\Seeder;
use App\\Models\\Category;
use App\\Models\\Product;

class DatabaseSeeder extends Seeder {
    public function run(): void {
        $c1 = Category::create(['name' => 'Kalung']);
        $c2 = Category::create(['name' => 'Gelang']);
        $c3 = Category::create(['name' => 'Cincin']);

        Product::create(['category_id' => $c1->id, 'sku' => 'KLG-001', 'name' => 'Kalung Emas Titanium Minimalis', 'price' => 125000, 'stock' => 15, 'minimum_stock' => 5]);
        Product::create(['category_id' => $c2->id, 'sku' => 'GLG-002', 'name' => 'Gelang Serut Mutiara Air Tawar', 'price' => 85000, 'stock' => 3, 'minimum_stock' => 5]);
        Product::create(['category_id' => $c3->id, 'sku' => 'CCN-003', 'name' => 'Cincin Perak Solitaire Permata', 'price' => 95000, 'stock' => 20, 'minimum_stock' => 5]);
    }
}
"""
with open("database/seeders/DatabaseSeeder.php", "w") as f:
    f.write(seeder_code)

print("--- [4/5] Membuat Resource Controller, Route, & Blade Views ---")

# Product Controller
controller_code = """<?php
namespace App\\Http\\Controllers;

use App\\Models\\Product;
use App\\Models\\Category;
use Illuminate\\Http\\Request;

class ProductController extends Controller {
    public function index(Request $request) {
        $search = $request->get('search');
        $cat_id = $request->get('category_id');
        $sort   = $request->get('sort', 'name');

        $query = Product::with('category');

        if ($search) {
            $query->where(function($q) use ($search) {
                $q->where('name', 'like', "%{$search}%")
                  ->orWhere('sku', 'like', "%{$search}%");
            });
        }

        if ($cat_id) {
            $query->where('category_id', $cat_id);
        }

        $sortColumn = in_array($sort, ['name', 'price', 'stock']) ? $sort : 'name';
        $products = $query->orderBy($sortColumn)->paginate(5)->withQueryString();
        $categories = Category::all();

        return view('products.index', compact('products', 'categories', 'search', 'cat_id', 'sort'));
    }

    public function store(Request $request) {
        $validated = $request->validate([
            'sku' => 'required|unique:products,sku',
            'name' => 'required|max:150',
            'category_id' => 'required|exists:categories,id',
            'price' => 'required|numeric|min:0',
            'stock' => 'required|integer|min:0',
            'minimum_stock' => 'required|integer|min:0',
        ]);

        Product::create($validated);
        return redirect()->route('products.index')->with('success', 'Produk berhasil ditambahkan!');
    }

    public function edit(Product $product) {
        $categories = Category::all();
        return view('products.edit', compact('product', 'categories'));
    }

    public function update(Request $request, Product $product) {
        $validated = $request->validate([
            'sku' => 'required|unique:products,sku,' . $product->id,
            'name' => 'required|max:150',
            'category_id' => 'required|exists:categories,id',
            'price' => 'required|numeric|min:0',
            'stock' => 'required|integer|min:0',
            'minimum_stock' => 'required|integer|min:0',
        ]);

        $product->update($validated);
        return redirect()->route('products.index')->with('success', 'Produk berhasil diperbarui!');
    }

    public function destroy(Product $product) {
        $product->delete();
        return redirect()->route('products.index')->with('success', 'Produk berhasil dihapus!');
    }
}
"""
with open("app/Http/Controllers/ProductController.php", "w") as f:
    f.write(controller_code)

# Routes
routes_code = """<?php
use Illuminate\\Support\\Facades\\Route;
use App\\Http\\Controllers\\ProductController;

Route::get('/', function () {
    return redirect()->route('products.index');
});

Route::resource('products', ProductController::class);
"""
with open("routes/web.php", "w") as f:
    f.write(routes_code)

# Blade Layout
os.makedirs("resources/views/layouts", exist_ok=True)
layout_code = """<!DOCTYPE html>
<html>
<head>
    <title>Inventory Aksesoris - Laravel P5</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, sans-serif; margin: 0; padding: 20px; background: #FAF6EE; color: #4A3E3D; }
        .nav { background: #EFE3CF; padding: 12px 20px; border-radius: 10px; color: #4A3E3D; border: 1px solid #E5D3B3; margin-bottom: 25px; display: flex; justify-content: space-between; align-items: center; }
        .card { background: #FFFDF9; padding: 20px; border-radius: 10px; border: 1px solid #EFE3CF; box-shadow: 0 2px 8px rgba(180,160,130,0.08); margin-bottom: 25px; }
        table { width: 100%; border-collapse: collapse; background: #FFFDF9; border-radius: 8px; overflow: hidden; }
        th, td { border: 1px solid #EFE3CF; padding: 12px; text-align: left; }
        th { background-color: #FFE699; color: #5C4100; font-weight: bold; }
        tr:nth-child(even) { background-color: #FAF6EE; }
        .btn { padding: 6px 12px; text-decoration: none; border-radius: 6px; font-weight: bold; font-size: 13px; border: none; cursor: pointer; display: inline-block; }
        .btn-yellow { background: #FFE89C; color: #5C4100; border: 1px solid #F5D061; }
        .btn-edit { background: #FFF2C2; color: #664D00; border: 1px solid #FFE699; }
        .btn-delete { background: #FAD4D4; color: #7A1C1C; border: 1px solid #F5B7B7; }
        .alert-success { color: #155724; background: #D4EDDA; padding: 12px; border-radius: 8px; margin-bottom: 15px; border: 1px solid #C3E6CB; }
        .alert-danger { color: #721C24; background: #F8D7DA; padding: 12px; border-radius: 8px; margin-bottom: 15px; border: 1px solid #F5C6CB; }
        input, select { padding: 8px 12px; border: 1px solid #DDD0BD; border-radius: 6px; background: #FFFDF9; }
    </style>
</head>
<body>
    <div class="nav">
        <span style="font-size:18px; font-weight:bold; color:#5C4B37;">✨ Aksesoris Store (Laravel Framework P5)</span>
    </div>
    @if(session('success'))
        <div class="alert-success">{{ session('success') }}</div>
    @endif
    @if($errors->any())
        <div class="alert-danger">
            <ul>@foreach($errors->all() as $error) <li>{{ $error }}</li> @endforeach</ul>
        </div>
    @endif
    @yield('content')
</body>
</html>
"""
with open("resources/views/layouts/app.blade.php", "w") as f:
    f.write(layout_code)

# Blade Index
os.makedirs("resources/views/products", exist_ok=True)
index_blade = """@extends('layouts.app')

@section('content')
<div class="card">
    <h3 style="margin-top:0; color:#5C4B37;">+ Tambah Produk Aksesoris Baru (Blade Form + CSRF)</h3>
    <form method="POST" action="{{ route('products.store') }}" style="display:flex; flex-wrap:wrap; gap:10px; align-items:flex-end;">
        @csrf
        <div><label style="font-size:12px; font-weight:bold;">SKU</label><br><input type="text" name="sku" value="{{ old('sku') }}" required style="width:110px;"></div>
        <div><label style="font-size:12px; font-weight:bold;">Nama Produk</label><br><input type="text" name="name" value="{{ old('name') }}" required style="width:180px;"></div>
        <div>
            <label style="font-size:12px; font-weight:bold;">Kategori</label><br>
            <select name="category_id" required>
                @foreach($categories as $cat)
                    <option value="{{ $cat->id }}">{{ $cat->name }}</option>
                @endforeach
            </select>
        </div>
        <div><label style="font-size:12px; font-weight:bold;">Harga (Rp)</label><br><input type="number" name="price" value="{{ old('price') }}" required style="width:110px;"></div>
        <div><label style="font-size:12px; font-weight:bold;">Stok</label><br><input type="number" name="stock" value="{{ old('stock', 10) }}" required style="width:70px;"></div>
        <div><label style="font-size:12px; font-weight:bold;">Min. Stok</label><br><input type="number" name="minimum_stock" value="5" required style="width:70px;"></div>
        <button type="submit" class="btn btn-yellow">+ Simpan</button>
    </form>
</div>

<div class="card">
    <form method="GET" action="{{ route('products.index') }}" style="display:flex; gap:10px;">
        <input type="text" name="search" placeholder="Cari nama / SKU..." value="{{ $search }}">
        <select name="category_id">
            <option value="">-- Semua Kategori --</option>
            @foreach($categories as $cat)
                <option value="{{ $cat->id }}" {{ $cat_id == $cat->id ? 'selected':'' }}>{{ $cat->name }}</option>
            @endforeach
        </select>
        <select name="sort">
            <option value="name" {{ $sort == 'name' ? 'selected':'' }}>Urutkan Nama</option>
            <option value="price" {{ $sort == 'price' ? 'selected':'' }}>Urutkan Harga</option>
            <option value="stock" {{ $sort == 'stock' ? 'selected':'' }}>Urutkan Stok</option>
        </select>
        <button type="submit" class="btn btn-yellow">Filter</button>
    </form>
</div>

<table>
    <tr>
        <th>SKU</th>
        <th>Nama Produk</th>
        <th>Kategori</th>
        <th>Harga</th>
        <th>Stok</th>
        <th>Aksi (Eloquent CRUD)</th>
    </tr>
    @forelse($products as $p)
    <tr>
        <td><b>{{ $p->sku }}</b></td>
        <td>{{ $p->name }}</td>
        <td>{{ $p->category->name }}</td>
        <td>Rp {{ number_format($p->price, 2) }}</td>
        <td><b>{{ $p->stock }}</b></td>
        <td>
            <a href="{{ route('products.edit', $p->id) }}" class="btn btn-edit">Edit</a>
            <form action="{{ route('products.destroy', $p->id) }}" method="POST" style="display:inline;">
                @csrf
                @method('DELETE')
                <button type="submit" class="btn btn-delete" onclick="return confirm('Hapus produk ini?')">Hapus</button>
            </form>
        </td>
    </tr>
    @empty
    <tr><td colspan="6" style="text-align:center;">Belum ada data produk.</td></tr>
    @endforelse
</table>

<div style="margin-top:15px; text-align:center;">
    {{ $products->links() }}
</div>
@endsection
"""
with open("resources/views/products/index.blade.php", "w") as f:
    f.write(index_blade)

# Blade Edit
edit_blade = """@extends('layouts.app')

@section('content')
<div class="card" style="max-width:500px; margin:0 auto;">
    <h2 style="margin-top:0; color:#5C4B37;">Edit Produk Aksesoris</h2>
    <form method="POST" action="{{ route('products.update', $product->id) }}">
        @csrf
        @method('PUT')
        <div style="margin-bottom:12px;">
            <label><b>SKU:</b></label><br>
            <input type="text" name="sku" value="{{ old('sku', $product->sku) }}" required style="width:100%;">
        </div>
        <div style="margin-bottom:12px;">
            <label><b>Nama Produk:</b></label><br>
            <input type="text" name="name" value="{{ old('name', $product->name) }}" required style="width:100%;">
        </div>
        <div style="margin-bottom:12px;">
            <label><b>Kategori:</b></label><br>
            <select name="category_id" required style="width:100%;">
                @foreach($categories as $cat)
                    <option value="{{ $cat->id }}" {{ $product->category_id == $cat->id ? 'selected':'' }}>{{ $cat->name }}</option>
                @endforeach
            </select>
        </div>
        <div style="margin-bottom:12px;">
            <label><b>Harga:</b></label><br>
            <input type="number" name="price" value="{{ old('price', $product->price) }}" required style="width:100%;">
        </div>
        <div style="margin-bottom:12px;">
            <label><b>Stok:</b></label><br>
            <input type="number" name="stock" value="{{ old('stock', $product->stock) }}" required style="width:100%;">
            <input type="hidden" name="minimum_stock" value="{{ $product->minimum_stock }}">
        </div>
        <button type="submit" class="btn btn-yellow" style="width:100%;">Perbarui Produk</button>
        <a href="{{ route('products.index') }}" style="display:block; text-align:center; margin-top:10px; color:#777; text-decoration:none;">Batal</a>
    </form>
</div>
@endsection
"""
with open("resources/views/products/edit.blade.php", "w") as f:
    f.write(edit_blade)

# Run Migration & Seeder
os.system("php artisan migrate:fresh --seed")

print("--- [5/5] Menjalankan Server Laravel & Cloudflare Tunnel ---")
subprocess.Popen(["php", "artisan", "serve", "--host=127.0.0.1", "--port=8000"])
time.sleep(2)

tunnel_proc = subprocess.Popen(
    ["../cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("\nMengambil tautan web (tunggu 5-10 detik)...")
web_url = None
for _ in range(25):
    line = tunnel_proc.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            web_url = match.group(0)
            break
    time.sleep(0.5)

if web_url:
    print("\n" + "="*60)
    print(" BUKA TAUTAN APLIKASI LARAVEL DI BAWAH INI:")
    print(f" {web_url}")
    print("="*60 + "\n")
else:
    print("\nGagal membaca URL tunnel secara otomatis.")

--- [1/5] Menginstal PHP, Composer, MySQL, & Cloudflare Tunnel ---
--- [2/5] Membuat Project Laravel (inventory-laravel) ---
--- [3/5] Membuat Migration & Model ---
--- [4/5] Membuat Resource Controller, Route, & Blade Views ---
--- [5/5] Menjalankan Server Laravel & Cloudflare Tunnel ---

Mengambil tautan web (tunggu 5-10 detik)...

 BUKA TAUTAN APLIKASI LARAVEL DI BAWAH INI:
 https://heads-spread-loop-channels.trycloudflare.com



# Dokumentasi Tugas P5: Inventory Management System (Laravel)

## 1. Arsitektur MVC Project
- **Model (`App\Models\Product` & `App\Models\Category`):** Mengelola tabel database dan relasi Eloquent (`belongsTo` & `hasMany`).
- **View (`resources/views/products/`):** Menampilkan antarmuka web interaktif dengan Blade Layout, CSRF protection, dan pesan validasi error.
- **Controller (`App\Http\Controllers\ProductController`):** Mengatur logika bisnis, menangani validasi input request, serta mengarahkan response.

---

## 2. Entity Relationship Diagram (ERD)

```text
  [ categories ]
  ├── id (PK)
  └── name
         │
         │ 1 (One-to-Many)
         ▼
  [ products ]
  ├── id (PK)
  ├── category_id (FK) -> categories.id
  ├── sku (UNIQUE)
  ├── name
  ├── price
  ├── stock
  └── minimum_stock

In [2]:
import os

print("--- Memperbaiki Hak Akses MySQL ---")
# Reset password/akses user root di MySQL agar bisa diakses oleh Laravel
fix_mysql_cmd = """
mysql -u root -e "
ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY '';
FLUSH PRIVILEGES;
"
"""
os.system(fix_mysql_cmd)

# Masuk ke folder proyek Laravel dan jalankan ulang migration
os.chdir("/content/inventory-laravel")
os.system("php artisan config:clear")
os.system("php artisan migrate:fresh --seed")

print("\n✅ Perbaikan berhasil!")
print("Silakan REFRESH / MUAT ULANG halaman web kamu di browser!")

--- Memperbaiki Hak Akses MySQL ---

✅ Perbaikan berhasil!
Silakan REFRESH / MUAT ULANG halaman web kamu di browser!
